# Authorization Code + PKCE login → a **Vortex** table

Sign in with the **OAuth2 Authorization Code grant with PKCE** (RFC 7636) — it opens your
browser and captures the redirect on a loopback socket (best when the notebook runs on your
own machine) — then register a Vortex generic table and round-trip data through Vortex.

> **Vortex note:** Vortex's Python object-store (S3) write path is newer and not yet wired
> into this SDK's credential mapping, so this notebook writes the `.vortex` file **locally**
> and registers the table's format with Lakekeeper. Swapping in a vended-S3 write is a
> follow-up once Vortex's object-store API stabilises.

> **Kernel:** run this with the **Python (pylakekeeper)** kernel (the `python/.venv`). If you
> see `ModuleNotFoundError: No module named 'pylakekeeper'`, you're on the wrong kernel — see
> `examples/README.md` → *Running the notebooks*.

## Requirements
```sh
cd python && pip install -e '.[examples]' vortex-data
```
A Keycloak-fronted Lakekeeper with an existing warehouse + namespace (see `examples/README.md`).

In [ ]:
import os

KEYCLOAK = os.environ.get("KEYCLOAK", "http://localhost:30080")
REALM = os.environ.get("KEYCLOAK_REALM", "iceberg")
ISSUER = f"{KEYCLOAK}/realms/{REALM}"
AUTHORIZATION_URL = f"{ISSUER}/protocol/openid-connect/auth"
TOKEN_URL = f"{ISSUER}/protocol/openid-connect/token"
CLIENT_ID = os.environ.get("OAUTH_CLIENT_ID", "lakekeeper")  # public client, PKCE, no secret
SCOPE = os.environ.get("OAUTH_SCOPE", "openid offline_access")

# --- Lakekeeper ---
LAKEKEEPER = os.environ.get("LAKEKEEPER", "http://localhost:8181")
WAREHOUSE_ID = os.environ.get("WAREHOUSE_ID", "")  # the warehouse UUID (URL prefix, not name)
PROJECT_ID = os.environ.get("PROJECT_ID") or None
NAMESPACE = os.environ.get("NAMESPACE", "ai.test")

In [ ]:
import base64
import json
import time


def token_seconds_left(auth_header: str) -> int:
    """Decode a Bearer JWT's `exp` claim and return seconds until it expires."""
    token = auth_header.split(" ", 1)[-1]
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)  # restore base64 padding
    claims = json.loads(base64.urlsafe_b64decode(payload))
    return int(claims["exp"] - time.time())

## Sign in (authorization code + PKCE)
Runs opens your browser; the cell returns after you sign in.

In [ ]:
from pylakekeeper import AuthorizationCodeFlow

auth = AuthorizationCodeFlow(
    authorization_url=AUTHORIZATION_URL,
    token_url=TOKEN_URL,
    client_id=CLIENT_ID,
    scope=SCOPE,
    # redirect_port=0 -> an OS-assigned free loopback port.
)
print("Signed in ✅  token expires in", token_seconds_left(auth.auth_header()), "s")

## Sample data

In [ ]:
import numpy as np
import pyarrow as pa

EMBED_DIM = 8
ROWS = 100
_rng = np.random.default_rng(42)
_embeddings = _rng.standard_normal((ROWS, EMBED_DIM)).astype(np.float32)

sample = pa.table(
    {
        "id": pa.array(range(ROWS), type=pa.int64()),
        "sku": pa.array([f"SKU-{i:06d}" for i in range(ROWS)]),
        "embedding": pa.FixedSizeListArray.from_arrays(
            pa.array(_embeddings.reshape(-1), type=pa.float32()), EMBED_DIM
        ),
    }
)
print(sample.schema)

## Register the Vortex table and write it (local)

We create the generic table with `format=vortex` in Lakekeeper (metadata + base location),
then write the `.vortex` file locally via the `vortex` library (see the note at the top on
why the write is local rather than to the vended S3 location).

In [ ]:
import tempfile
from pathlib import Path

import vortex as vx

from pylakekeeper import Client, ConflictError, GenericTableFormat

TABLE = os.environ.get("TABLE", "vortex_embeddings")

with Client(base_url=LAKEKEEPER, warehouse=WAREHOUSE_ID, auth=auth, project_id=PROJECT_ID) as c:
    try:
        c.generic_tables.create(
            NAMESPACE,
            TABLE,
            format=GenericTableFormat.VORTEX,
            properties={"embedding-dim": str(EMBED_DIM)},
        )
    except ConflictError:
        print(f"{NAMESPACE}.{TABLE} already exists — continuing")

    resp = c.generic_tables.load(NAMESPACE, TABLE)
    print("registered vortex table at base location:", resp.location)

# Write the Vortex file locally (kept in `vortex_path` for the read cell below).
vortex_path = str(Path(tempfile.gettempdir()) / f"{TABLE}.vortex")
vx.io.write(vx.array(sample), vortex_path)
print("wrote", vortex_path)

## Read the Vortex data back

`vx.io.read_url` returns a Vortex array; `.to_arrow_table()` hands it back as an Arrow table.
Vortex reads are lazy/columnar, so you can project columns (and push down row filters) instead
of materialising everything.

In [ ]:
table = vx.io.read_url(vortex_path).to_arrow_table()
print("row_count =", table.num_rows)
print(table.schema)
print(table.select(["id", "sku"]).slice(0, 5))

# Column projection: read just what you need.
ids_only = vx.io.read_url(vortex_path, projection=["id", "sku"]).to_arrow_table()
print("\nprojected columns:", ids_only.column_names)

## The session outlives the access token
Force an expiry — the refresh token renews silently, no second browser prompt.

In [ ]:
before = token_seconds_left(auth.auth_header())
print("current access token expires in", before, "s")

auth.invalidate()  # what the transport does on a 401 — simulate expiry

after = token_seconds_left(auth.auth_header())  # no re-login: silent refresh
print("after refresh, new token expires in", after, "s")
assert after >= before, "expected a freshly-minted token"
print("\n✅ Session renewed via refresh_token — no re-authentication.")